# EXP-025 — Complete FP16 VLM–CLIP probe (TASK-027B)

Upload the existing TASK-025C `exp023_handoff.zip` to MyDrive, select a **Colab L4**, set `AUTHORIZE_EXP025=True` once, then **Runtime → Run all**. Python3.12.10, model revisions, prompt and dependencies stay pinned. Models/images stay in /content; completed per-image caches, fold outputs and logs persist on Drive. No new data ZIP is needed.

The notebook runs NF4 and FP16 on the **same existing30 paintings with the same fixed prompt**, shows images and descriptions side by side with measured latency/VRAM, and **pauses exactly once for your visual assessment**. Approve broadly useful semantics even with isolated mistakes or nationality wording. Reject only serious systematic scene hallucination. Warnings remain visible and saved. Technical failures (corrupt/missing embeddings, lost tokens, invalid/nonfinite outputs) still stop. No prompt revisions or new models are launched.

**After approval:** automatically complete FP16 extraction for all1335 paintings and **all five P2 grouped folds**, with visual-only, semantic-only, early/late fusion, shuffled controls and fixed E50+L reference. Bootstrap statistics are evidence, never continuation gates. After rejection: preserve/export smoke and stop; direct hidden states/stronger models are recommendations only. Exact approved smoke and completed folds resume without another review. Changed smoke records need a new assessment.

The review pause allows up to10 minutes within the configured allocation; no response is never approval. MAX_HOURS is a maximum including setup/review, not a target. Export/download is attempted before release on success or controlled failure; abrupt VM loss preserves only completed persistent boundaries. API returns do not prove browser arrival or physical GPU deallocation. Built with Qwen under its non-commercial Research License; competition eligibility remains unconfirmed. E50+L stays production. No Development gold, confirmation labels, fine-tuning or Codabench upload.


In [ ]:
# OWNER CONFIGURATION: authorize once, then Runtime -> Run all.
AUTHORIZE_EXP025 = False
INPUT_ZIP = "/content/drive/MyDrive/exp023_handoff.zip"
PERSISTENT_ROOT = "/content/drive/MyDrive/exp025"
LOCAL_ROOT = "/content/exp025b"
MAX_HOURS = 2.0
AUTO_DOWNLOAD = True
AUTO_RELEASE = True
SOURCE_REVISION = "fc3ec336ca99871fa87d1d51c4b1ddfedb208c86"
ZIP_SHA256 = "2db4a53679940c7eb63c7c8272b91be8ec0618bb6c04a69ebaa15909e20f64e0"


In [ ]:
import csv, hashlib, json, os, shutil, subprocess, sys, time, traceback, zipfile
from pathlib import Path
from google.colab import drive, files, runtime
started = time.time()
local, persistent = Path(LOCAL_ROOT), Path(PERSISTENT_ROOT)
attempt = persistent / "attempts" / time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
result = {"status":"STARTING", "research_only":True, "production":"E50+L unchanged",
          "browser_arrival":"UNVERIFIED", "physical_GPU_deallocation":"UNVERIFIED"}
mounted = False

def run(command, name, env=None):
    remaining = started + MAX_HOURS*3600 - time.time() - 120
    if remaining <= 0: raise TimeoutError("Allocation exhausted; completed semantic records persist")
    print("Running", name, flush=True)
    log = attempt / (name + ".log")
    with log.open("w") as handle:
        process = subprocess.run(command, stdout=handle, stderr=subprocess.STDOUT, timeout=remaining, env=env)
    tail = log.read_text(errors="replace").splitlines()[-16:]
    print("\n".join(tail))
    if process.returncode: raise RuntimeError(f"{name} failed; see {log}. Cached completed images are retained.")


def show_comparison(directory):
    from IPython.display import HTML, display
    report = json.loads((directory/"smoke_comparison.json").read_text())
    print(json.dumps({k:report[k] for k in ("NF4","FP16","performance_comparison")},indent=2))
    print("Warnings are retained below; isolated inaccuracies or nationality wording are not vetoes.")
    display(HTML((directory/"smoke_comparison.html").read_text()))

def one_smoke_review():
    # Bound the single visual-review pause on Colab's Linux main thread.
    import signal, threading
    remaining = int(started + MAX_HOURS*3600 - time.time() - 120)
    if remaining <= 0: raise TimeoutError("Budget expired before smoke review; results are preserved")
    armed = hasattr(signal,"SIGALRM") and threading.current_thread() is threading.main_thread()
    def expired(*args):raise TimeoutError("Smoke review timed out; no approval inferred, results are preserved")
    previous = signal.signal(signal.SIGALRM,expired) if armed else None
    if armed:signal.alarm(min(600,remaining))
    try:
        return input("ONE SMOKE REVIEW: type APPROVE if FP16 preserves broadly useful visual semantics despite isolated errors; type REJECT for serious systematic scene hallucination. Optional reason after a colon: ")
    finally:
        if armed:signal.alarm(0);signal.signal(signal.SIGALRM,previous)

def probe_stages(python, source, inputs, persistent, attempt, env):
    fp16_cache, nf4_cache = persistent/"semantics", persistent/"nf4_l4_smoke"
    base = [python,str(source/"scripts/extract_exp025.py"),"--inputs",str(inputs),"--model-cache",str(local/"models")]
    remaining = lambda:str(max(61,int(started+MAX_HOURS*3600-time.time()-120)))
    run([*base,"--config",str(source/"configs/exp025_nf4.json"),"--cache",str(nf4_cache),"--max-seconds",remaining()],"nf4_smoke",env)
    run([*base,"--cache",str(fp16_cache),"--max-seconds",remaining()],"fp16_smoke",env)
    comparison_dir = attempt/"smoke"
    worker = "from pathlib import Path; import sys,json; sys.path.insert(0,sys.argv[1]); from st3.vlm_smoke_review import build_comparison; root=Path(sys.argv[2]); build_comparison(Path(sys.argv[3]),Path(sys.argv[4]),Path(sys.argv[5]),json.loads((root/'configs/exp025_nf4.json').read_text()),json.loads((root/'configs/exp025.json').read_text()),Path(sys.argv[6]))"
    run([python,"-c",worker,str(source/"src"),str(source),str(inputs),str(nf4_cache),str(fp16_cache),str(comparison_dir)],"smoke_comparison",env)
    comparison = json.loads((comparison_dir/"smoke_comparison.json").read_text())
    if comparison["technical_status"]!="PASS":raise RuntimeError("Smoke technical validation failed")
    show_comparison(comparison_dir)
    signature = json.dumps(comparison["fp16_smoke"],sort_keys=True).encode()
    review_file = persistent/"reviews"/(hashlib.sha256(signature).hexdigest()+".json")
    if review_file.exists():
        review = json.loads(review_file.read_text())
        if review["fp16_smoke"]!=comparison["fp16_smoke"]:raise ValueError("Existing review does not match the current smoke")
        print("Reusing exact-smoke owner review:",review["decision"])
    else:
        answer = one_smoke_review()
        choice,_,note = answer.partition(":")
        decision = {"APPROVE":"BROADLY_USABLE","REJECT":"SYSTEMATIC_SCENE_HALLUCINATION"}.get(choice.strip().upper())
        if decision is None:raise ValueError("No explicit smoke assessment; preserved results, no full extraction")
        review = {"decision":decision,"note":note.strip(),"timestamp":time.time(),"fp16_smoke":comparison["fp16_smoke"],
                  "comparison_sha256":hashlib.sha256((comparison_dir/"smoke_comparison.json").read_bytes()).hexdigest(),
                  "criterion":"Broad visual semantics; isolated errors/wording are warnings, not rejection criteria"}
        review_file.parent.mkdir(parents=True,exist_ok=True)
        temporary = review_file.with_suffix(".partial");temporary.write_text(json.dumps(review,indent=2));temporary.replace(review_file)
    if review["decision"]=="SYSTEMATIC_SCENE_HALLUCINATION":
        return {"status":"STOPPED_SYSTEMATIC_HALLUCINATION","decision":"ALTERNATIVE_VLM_REPRESENTATION_NEEDED",
                "recommendation":"Consider direct VLM hidden-state features or a stronger model in a separately authorized task. Neither is launched.","review_file":str(review_file)}
    if review["decision"]!="BROADLY_USABLE":raise ValueError("Invalid stored smoke decision")
    # No further owner pauses, prompt changes, semantic-warning vetoes or CV gates.
    run([*base,"--cache",str(fp16_cache),"--full","--review-file",str(review_file),"--max-seconds",remaining()],"full_extraction",env)
    head_dir = persistent/("heads_fp16_"+SOURCE_REVISION[:12])
    run([python,str(source/"scripts/run_exp025.py"),"--inputs",str(inputs),"--cache",str(fp16_cache),"--run-dir",str(head_dir)],"five_fold_heads",env)
    completed = json.loads((head_dir/"completed.json").read_text())
    if completed["status"]!="FIVE_FOLD_COMPLETE" or completed["folds"]!=[1,2,3,4,5]:raise ValueError("All five folds are mandatory")
    print("Five-fold results:\n",(head_dir/"summary.csv").read_text())
    print("True-semantic incremental evidence:\n",(head_dir/"contribution_evidence.json").read_text())
    return {"status":"FIVE_FOLD_COMPLETE","review_file":str(review_file),"heads":str(head_dir),"promotion":False}

try:
    drive.mount("/content/drive");mounted = True
    attempt.mkdir(parents=True, exist_ok=False);local.mkdir(parents=True, exist_ok=True)
    if not AUTHORIZE_EXP025: raise RuntimeError("Set AUTHORIZE_EXP025=True in Configuration before the authorized Run all")
    if not (0 < MAX_HOURS < float("inf")): raise ValueError("MAX_HOURS must be positive and finite")
    with Path(INPUT_ZIP).open("rb") as handle:
        if hashlib.file_digest(handle,"sha256").hexdigest() != ZIP_SHA256: raise ValueError("Use the exact TASK-025C private input ZIP; SHA256 mismatch")
    source = local / "source"
    if not source.exists():
        run(["git","clone","--no-checkout","https://github.com/bqa100507-spec/SemEval-2027-Task-12---ArtEmotion-28-Subtask-3---Image-Only-Cross-Cultural-Sentiment-Prediction.git",str(source)],"clone")
        run(["git","-C",str(source),"checkout","--detach",SOURCE_REVISION],"checkout")
    if subprocess.check_output(["git","-C",str(source),"rev-parse","HEAD"],text=True).strip()!=SOURCE_REVISION:
        raise ValueError("Existing local source revision differs; choose a fresh LOCAL_ROOT")
    expected = json.loads((source/"configs/exp024.json").read_text())["input_sha256"]
    inputs = local / "inputs"
    with zipfile.ZipFile(INPUT_ZIP) as archive:
        protocol = archive.read("training/protocol.csv").decode("utf-8")
        rows = [row for row in csv.DictReader(protocol.splitlines()) if row["role"]=="development"]
        if len(rows)!=1335: raise ValueError("Expected1335 P2 training paintings")
        targets = {**expected, **{"artwork/images/"+row["image"]:row["image_sha256"] for row in rows}}
        for name, checksum in targets.items():
            target = (inputs/name).resolve()
            if not target.is_relative_to(inputs.resolve()): raise ValueError("Unsafe input ZIP path")
            if target.is_file():
                with target.open("rb") as handle: current=hashlib.file_digest(handle,"sha256").hexdigest()
                if current!=checksum: raise ValueError("Existing local input changed")
                continue
            raw = archive.read(name)
            if hashlib.sha256(raw).hexdigest()!=checksum: raise ValueError("Input checksum mismatch: "+name)
            target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
    env = dict(os.environ, UV_PYTHON_INSTALL_DIR=str(local/"python"),UV_CACHE_DIR=str(local/"uv_cache"),HF_HUB_DISABLE_XET="1")
    run([sys.executable,"-m","pip","install","uv==0.7.13"],"uv",env)
    run([sys.executable,"-m","uv","python","install","3.12.10"],"python",env)
    if not (local/"venv").exists():run([sys.executable,"-m","uv","venv","--python","3.12.10",str(local/"venv")],"venv",env)
    python = str(local/"venv/bin/python")
    run([sys.executable,"-m","uv","pip","sync","--python",python,"--extra-index-url","https://download.pytorch.org/whl/cu126","--index-strategy","unsafe-best-match",str(source/"requirements-exp025.lock.txt")],"dependencies",env)
    run([sys.executable,"-m","uv","pip","check","--python",python],"dependency_check",env)
    run([python,"-c","import torch; assert torch.cuda.is_available(), 'Select CUDA GPU'; print(torch.cuda.get_device_name()); assert 'L4' in torch.cuda.get_device_name(), 'Select an L4 GPU for the matched precision smoke'; assert torch.cuda.mem_get_info()[0]>10*1024**3, 'L4 needs at least10GiB free VRAM'"],"GPU_check",env)
    result.update(probe_stages(python,source,inputs,persistent,attempt,env))
except Exception as error:
    result.update(status="TECHNICAL_OR_EXECUTION_STOP",error=str(error),traceback=traceback.format_exc())
finally:
    try:
        if mounted:
            attempt.mkdir(parents=True,exist_ok=True)
            (attempt/"status.json").write_text(json.dumps(result,indent=2))
            for report in [*(persistent/"semantics").glob("*/extraction_report.json"),*(persistent/"nf4_l4_smoke").glob("*/extraction_report.json")]:
                print("Measured extraction report:",report.read_text())
            head_summary = persistent/("heads_fp16_"+SOURCE_REVISION[:12])/"summary.csv"
            if head_summary.exists():print("Head results:\n",head_summary.read_text())
            output = attempt/"outputs.zip"
            with zipfile.ZipFile(output,"w",zipfile.ZIP_DEFLATED) as archive:
                for directory,prefix in [(attempt,"attempt"),(persistent/"semantics","semantics"),(persistent/("heads_fp16_"+SOURCE_REVISION[:12]),"heads"),(persistent/"nf4_l4_smoke","nf4_smoke"),(persistent/"reviews","reviews")]:
                    if directory.exists():
                        for path in directory.rglob("*"):
                            if path.is_file() and path!=output and not path.name.endswith(".partial"):
                                archive.write(path,prefix+"/"+path.relative_to(directory).as_posix())
            with zipfile.ZipFile(output) as archive:
                if archive.testzip():raise IOError("Output ZIP read-back failed")
            with output.open("rb") as handle:checksum=hashlib.file_digest(handle,"sha256").hexdigest()
            (attempt/"outputs.sha256").write_text(checksum+"  outputs.zip\n")
            result["persistent_readback"]="PASS"
            if AUTO_DOWNLOAD:
                try:files.download(str(output));result["download_request"]="API_RETURNED"
                except Exception as error:result["download_request"]="FAILED: "+str(error)
    except Exception as error:result["export_error"]=str(error)
    finally:
        result["runtime_release"]="REQUESTED_NOT_CONFIRMED" if AUTO_RELEASE else "DISABLED"
        if mounted:
            try:(attempt/"final_report.json").write_text(json.dumps(result,indent=2))
            except Exception:pass
        print(json.dumps(result,indent=2))
        if AUTO_RELEASE:
            try:runtime.unassign()
            except Exception as error:print("Runtime release request failed:",error)
